# Step 2 — Long-video 30min / 60min comparison

Scans `DATA_DIR` for every `*_combined.csv`, classifies it as either a `30min` or `60min` recording from the filename, computes the trial-mean of the four behavioral signals, and writes two summary CSVs (one per condition).

**Filename parsing**:
- date: `\d{4}_\d{4}` at the start of the basename, e.g. `2026_0407`
- fly: `_Fly\d+`, e.g. `Fly4`
- condition: case-insensitive substring `30min` or `60min` anywhere in the filename

Files missing any of those tags are skipped with a console warning.

**Output CSVs** (saved in `DATA_DIR`):
- `longvid_30min_summary.csv` — one column per `30min` file, identified by `<date>_Fly<n>_30min`.
- `longvid_60min_summary.csv` — same for `60min` files.

Each CSV has 5 rows: header + 4 metric rows (avg wingbeat amplitude, avg wingbeat frequency, avg inferred flight power, avg spiracle aperture).

---

## 中文说明

**用途**：比较"30 分钟"与"60 分钟"两种长视频飞行记录条件下的行为差异。它扫描 `DATA_DIR` 中的每个 `*_combined.csv`，根据文件名判定其属于 `30min` 还是 `60min` 条件，计算四个行为信号在指定时间窗内的实验均值，并按条件分别汇总；随后做 30min vs 60min 的配对比较、气门与各飞行信号的相关性、以及时间序列叠图。

**预期输入**：
- 文件类型：`DATA_DIR` 下的 `*_combined.csv`（Step 1 输出）。
- 必需列：`time_s`（NIDAQ 绝对时间，注意首样本通常在 ~9.5 s 而非 0）、`rescaled_intensity`（气门开度）、`wingbeat_amplitude`（rad）、`wingbeat_frequency`（Hz）、`inferred_flight_power`（W/kg，由翅拍幅度与频率推算）。
- 文件名约定（在基础名之内解析出三项标签）：开头的日期 `\d{4}_\d{4}`（如 `2026_0407`）、`_Fly\d+`（如 `Fly4`）、以及文件名任意位置不区分大小写的 `30min` 或 `60min` 子串。缺任一标签的文件会被告警并跳过。

**预期输出**（均写入 `DATA_DIR`）：
- `longvid_30min_summary.csv` / `longvid_60min_summary.csv`：每个文件一列（列名 `<date>_Fly<n>_<条件>`），4 个指标各一行。
- `longvid_30min_60min_paired_comparison.svg` / `.png`：30min vs 60min 配对箱线图（Block 4）。
- `longvid_30min_correlations.csv` / `longvid_60min_correlations.csv`：气门 vs 各飞行信号的相关系数（Block 5）。
- `longvid_10_70s_overlay.svg` / `.png`：两条件、四指标的时间序列叠图（Block 6）。

**注意**：Block 3–5 使用的分析窗为 `T_MIN_S`–`T_MAX_S`；Block 6 的绘图窗 `T_PLOT_MIN_S`–`T_PLOT_MAX_S` 独立设置。

### Block 1 — Imports & parameters

Loads libraries (and `set_plot_style` from `spiracle_helper_functions` for editable-text SVGs) and sets the run parameters: `DATA_DIR`, the output filenames, the analysis time window `[T_MIN_S, T_MAX_S]` in NIDAQ absolute seconds (used by Blocks 3–5), and the `METRICS` list of `(column, human label)` pairs.

**中文**：导入所需库（并从 `spiracle_helper_functions` 引入 `set_plot_style`，以生成可编辑文字的 SVG），并设置运行参数：`DATA_DIR`、输出文件名、以 NIDAQ 绝对秒计的分析时间窗 `[T_MIN_S, T_MAX_S]`（Block 3–5 使用），以及 `(列名, 可读标签)` 形式的指标列表 `METRICS`。

In [1]:
# Block 1 — imports + parameters
import re
from pathlib import Path

import numpy as np
import pandas as pd

from spiracle_helper_functions import set_plot_style
set_plot_style()   # Arial + editable-text SVG, 300 dpi (pipeline default)

# Folder to scan
DATA_DIR  = Path(r"C:\Users\Lylah\Desktop\data_processing\0_flight_30min_vs_60min")
RECURSIVE = False

# Output filenames (in DATA_DIR)
OUT_30_NAME = "longvid_30min_summary.csv"
OUT_60_NAME = "longvid_60min_summary.csv"

# Restrict per-file analysis to time_s in [T_MIN_S, T_MAX_S] (inclusive) of each
# combined.csv. `time_s` is NIDAQ absolute time, not 0-based — the first sample
# is wherever the Phantom/Basler triggers started (typically ~9.5 s).
# Set either bound to None to leave that side open.
T_MIN_S = 10.0
T_MAX_S = 70.0

# Metrics: (column in <stem>_combined.csv, human-readable row label)
METRICS = [
    ("wingbeat_amplitude",    "avg wingbeat amplitude (rad)"),
    ("wingbeat_frequency",    "avg wingbeat frequency (Hz)"),
    ("inferred_flight_power", "avg inferred flight power (W/kg)"),
    ("rescaled_intensity",    "avg spiracle aperture"),
]


h:\.shortcut-targets-by-id\10pxdlRXtzFB-abwDGi0jOGOFFNm3pmFK\Tuthill Lab Shared\Yichen\Spiracle\Analysis Code\spiracle_helper_functions.py:5: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.4.4)
  from scipy.io import loadmat


### Block 2 — Filename parsing & per-file summary

`parse_basename` extracts `(date, fly_num, condition)` from a filename using the regexes above, returning `None` if any tag is missing. `load_and_summarize` reads one combined CSV, restricts to `[t_min_s, t_max_s]`, and returns the nanmean of each metric column (NaN for missing columns).

**中文**：`parse_basename` 用上面的正则从文件名中提取 `(日期, 果蝇编号, 条件)`，缺任一标签则返回 `None`。`load_and_summarize` 读取单个合并 CSV，限制到 `[t_min_s, t_max_s]` 时间窗，并返回每个指标列的 nanmean（缺列则为 NaN）。

In [2]:
# Block 2 — filename parsing + per-file summary

DATE_RE  = re.compile(r"^(\d{4}_\d{4})")
FLY_RE   = re.compile(r"_Fly(\d+)", re.IGNORECASE)
COND_30  = re.compile(r"30min",      re.IGNORECASE)
COND_60  = re.compile(r"60min",      re.IGNORECASE)


def parse_basename(name: str):
    """Return (date, fly_num, condition) or None if any field is missing."""
    stem = name
    if stem.endswith("_combined.csv"):
        stem = stem[: -len("_combined.csv")]
    elif stem.endswith(".csv"):
        stem = stem[: -len(".csv")]

    date_m = DATE_RE.search(stem)
    fly_m  = FLY_RE.search(stem)
    if COND_30.search(stem):
        cond = "30min"
    elif COND_60.search(stem):
        cond = "60min"
    else:
        cond = None

    if date_m is None or fly_m is None or cond is None:
        return None
    return date_m.group(1), int(fly_m.group(1)), cond


def load_and_summarize(path: Path, t_min_s=None, t_max_s=None):
    """Read combined.csv, return {col: nanmean} for each metric col within
    time_s in [t_min_s, t_max_s] (inclusive). Either bound can be None to
    leave that side open. Missing column -> NaN.
    """
    df = pd.read_csv(path)
    if "time_s" in df.columns:
        if t_min_s is not None:
            df = df.loc[df["time_s"] >= t_min_s]
        if t_max_s is not None:
            df = df.loc[df["time_s"] <= t_max_s]
    out = {}
    for col, _ in METRICS:
        if col in df.columns:
            arr = pd.to_numeric(df[col], errors="coerce").values
            out[col] = float(np.nanmean(arr)) if np.isfinite(arr).any() else np.nan
        else:
            out[col] = np.nan
    return out

### Block 3 — Runner: scan, group by condition, build per-condition CSVs

Scans every combined CSV, parses each filename, and buckets the per-file metric means into `data_30` / `data_60` keyed by `(date, fly)` (averaging if a fly has multiple files). `build_summary` turns a bucket into a DataFrame — a `metric` column plus one column per fly — and both are written to the two summary CSVs.

**中文**：扫描每个合并 CSV，解析文件名，并按 `(日期, 果蝇)` 把各文件的指标均值分入 `data_30` / `data_60` 两个桶（同一果蝇有多个文件则取平均）。`build_summary` 把一个桶转成 DataFrame——一列 `metric` 加上每只果蝇一列——随后两者分别写入两个汇总 CSV。

In [3]:
# Block 3 — runner: scan, group by condition, build per-condition CSVs

files = sorted(
    DATA_DIR.rglob("*_combined.csv") if RECURSIVE else DATA_DIR.glob("*_combined.csv")
)
window_str = f"time_s in [{T_MIN_S}, {T_MAX_S}] s NIDAQ"
print(f"Found {len(files)} combined CSV(s) under {DATA_DIR}  (analysis window: {window_str})")

# (date, fly_num) -> list of (filename, {metric: mean})
data_30 = {}
data_60 = {}

for p in files:
    parsed = parse_basename(p.name)
    if parsed is None:
        print(f"  [skip] {p.name} -- can't parse date/fly/condition")
        continue
    date, fly_num, cond = parsed
    try:
        metrics = load_and_summarize(p, t_min_s=T_MIN_S, t_max_s=T_MAX_S)
    except Exception as e:
        print(f"  [skip] {p.name}: {type(e).__name__}: {e}")
        continue
    bucket = data_30 if cond == "30min" else data_60
    bucket.setdefault((date, fly_num), []).append((p.name, metrics))


def build_summary(bucket, condition_label: str) -> pd.DataFrame:
    """Return a 4-data-row DataFrame: 'metric' column + one column per (date, fly)."""
    columns = {"metric": [m_label for _, m_label in METRICS]}
    if not bucket:
        print(f"  [warn] No {condition_label} files found")
        return pd.DataFrame(columns)

    for (date, fly_num) in sorted(bucket.keys()):
        entries = bucket[(date, fly_num)]
        col_name = f"{date}_Fly{fly_num}_{condition_label}"
        if len(entries) > 1:
            print(f"  [note] {condition_label}: {len(entries)} files match {date}_Fly{fly_num} -- averaging")
            for fname, _ in entries:
                print(f"           {fname}")
        col_vals = []
        for col, _ in METRICS:
            vals = [m[col] for _, m in entries if np.isfinite(m[col])]
            col_vals.append(float(np.mean(vals)) if vals else np.nan)
        columns[col_name] = col_vals
    return pd.DataFrame(columns)


df_30 = build_summary(data_30, "30min")
df_60 = build_summary(data_60, "60min")

out_30_path = DATA_DIR / OUT_30_NAME
out_60_path = DATA_DIR / OUT_60_NAME
df_30.to_csv(out_30_path, index=False)
df_60.to_csv(out_60_path, index=False)

print(
    f"\nsaved: {out_30_path}  ({df_30.shape[0]} rows x {df_30.shape[1]} cols, "
    f"{df_30.shape[1] - 1} files)"
)
print(
    f"saved: {out_60_path}  ({df_60.shape[0]} rows x {df_60.shape[1]} cols, "
    f"{df_60.shape[1] - 1} files)"
)

for label, df in [("30min", df_30), ("60min", df_60)]:
    cols_data = [c for c in df.columns if c != "metric"]
    print(f"\n{label}: {len(cols_data)} file(s)")
    for c in cols_data:
        print(f"  {c}")

Found 14 combined CSV(s) under C:\Users\Lylah\Desktop\data_processing\0_flight_30min_vs_60min  (analysis window: time_s in [10.0, 70.0] s NIDAQ)

saved: C:\Users\Lylah\Desktop\data_processing\0_flight_30min_vs_60min\longvid_30min_summary.csv  (4 rows x 8 cols, 7 files)
saved: C:\Users\Lylah\Desktop\data_processing\0_flight_30min_vs_60min\longvid_60min_summary.csv  (4 rows x 8 cols, 7 files)

30min: 7 file(s)
  2026_0423_Fly2_30min
  2026_0427_Fly3_30min
  2026_0429_Fly6_30min
  2026_0429_Fly7_30min
  2026_0505_Fly1_30min
  2026_0505_Fly2_30min
  2026_0505_Fly3_30min

60min: 7 file(s)
  2026_0423_Fly2_60min
  2026_0427_Fly3_60min
  2026_0429_Fly6_60min
  2026_0429_Fly7_60min
  2026_0505_Fly1_60min
  2026_0505_Fly2_60min
  2026_0505_Fly3_60min


### Block 4 — Paired-comparison box plot (30min vs 60min)

Matches `(date, fly)` pairs that have BOTH conditions, then for each of the four metrics draws a 2×2 panel: box plots + jittered dots + paired connecting lines (30min → 60min for the same fly). Each panel runs a paired Wilcoxon and paired t-test and annotates the p-values. Saves `longvid_30min_60min_paired_comparison.svg/.png`.

**中文**：先匹配出同时具备两种条件的 `(日期, 果蝇)` 配对，再为四个指标各画一个 2×2 子图：箱线图 + 抖动散点 + 配对连线（同一果蝇的 30min → 60min）。每个子图运行配对 Wilcoxon 与配对 t 检验并标注 p 值。保存为 `longvid_30min_60min_paired_comparison.svg/.png`。

In [4]:
# Block 4 — paired-comparison box plot: 30min vs 60min for matched (date, fly) pairs
# Pipeline-style box plot + paired connecting lines + jittered dots.
# Paired Wilcoxon + paired t-test on each metric.

import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, ttest_rel

# y-axis limits per metric column. Keys match column names in <stem>_combined.csv.
# A clipping warning is printed if any data point falls outside the range.
YLIMS = {
    "rescaled_intensity":    (0.0, 1.0),     # spiracle aperture (0 - 100% as fraction)
    "wingbeat_amplitude":    (0.0, 5.5),
    "wingbeat_frequency":    (0.0, 250.0),
    "inferred_flight_power": (0.0, 300.0),
}

# ---- Match (date, fly) pairs that have BOTH conditions ----
pair_keys = sorted(set(data_30.keys()) & set(data_60.keys()))
print(f"Matched (date, fly) pairs with both 30min and 60min: {len(pair_keys)}")
for date, fly_num in pair_keys:
    print(f"  {date}_Fly{fly_num}")

if len(pair_keys) == 0:
    raise RuntimeError(
        "No matched pairs between data_30 and data_60. Did Block 3 finish, "
        "and does the folder actually contain a matching 30min and 60min file "
        "for at least one (date, fly)?"
    )


def _pair_value(bucket, key, col):
    """Mean of `col` across files matching `key` in `bucket`."""
    entries = bucket[key]
    vals = [m[col] for _, m in entries if np.isfinite(m[col])]
    return float(np.mean(vals)) if vals else np.nan


# metric -> (vals_30, vals_60), aligned to pair_keys
paired = {}
for col, _ in METRICS:
    v30 = np.array([_pair_value(data_30, k, col) for k in pair_keys], dtype=float)
    v60 = np.array([_pair_value(data_60, k, col) for k in pair_keys], dtype=float)
    paired[col] = (v30, v60)


def _paired_test(x, y, fn):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    n = int(m.sum())
    if n < 5:
        return np.nan, np.nan, n
    try:
        res = fn(x[m], y[m])
        return float(res.statistic), float(res.pvalue), n
    except Exception as e:
        print(f"  [warn] {fn.__name__} failed: {type(e).__name__}: {e}")
        return np.nan, np.nan, n


def _stars(p):
    if p is None or not np.isfinite(p): return "n.s."
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "n.s."


# Per-panel colors (same palette used elsewhere in pipeline box-plot blocks).
_PALETTE = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd",
            "#ff7f0e", "#17becf", "#8c564b", "#e377c2"]

# ---- 2x2 figure: one panel per metric ----
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
axes_flat = axes.flatten()
rng = np.random.default_rng(0)

stats_summary = []  # (label, w_stat, w_p, t_stat, t_p, n)
for i, (ax, (col, label)) in enumerate(zip(axes_flat, METRICS)):
    v30, v60 = paired[col]
    m = np.isfinite(v30) & np.isfinite(v60)
    x_v = v30[m]
    y_v = v60[m]

    color = _PALETTE[i % len(_PALETTE)]
    positions = np.array([0, 1])
    bp = ax.boxplot(
        [x_v, y_v], positions=positions, widths=0.55,
        showfliers=False, patch_artist=True,
        medianprops=dict(color="black", linewidth=1.2),
    )
    for patch in bp["boxes"]:
        patch.set_facecolor(color)
        patch.set_alpha(0.35)
        patch.set_edgecolor("black")
        patch.set_linewidth(0.8)
    for element in ("whiskers", "caps"):
        for line in bp[element]:
            line.set_color("black")
            line.set_linewidth(0.8)

    xs_a = positions[0] + rng.uniform(-0.08, 0.08, size=len(x_v))
    xs_b = positions[1] + rng.uniform(-0.08, 0.08, size=len(y_v))
    ax.scatter(xs_a, x_v, color=color, edgecolor="black", linewidth=0.5,
               s=32, zorder=3)
    ax.scatter(xs_b, y_v, color=color, edgecolor="black", linewidth=0.5,
               s=32, zorder=3)
    for xa, ya, xb, yb in zip(xs_a, x_v, xs_b, y_v):
        ax.plot([xa, xb], [ya, yb], color="gray", alpha=0.4, lw=0.7, zorder=2)

    # Stats
    w_stat, w_p, n = _paired_test(v30, v60, wilcoxon)
    t_stat, t_p, _ = _paired_test(v30, v60, ttest_rel)
    stats_summary.append((label, w_stat, w_p, t_stat, t_p, n))

    pw_str = f"p_w = {w_p:.3g} {_stars(w_p)}" if np.isfinite(w_p) else "p_w = n/a"
    pt_str = f"p_t = {t_p:.3g} {_stars(t_p)}" if np.isfinite(t_p) else "p_t = n/a"
    ax.text(0.02, 0.98, f"{pw_str}\n{pt_str}\nn = {n}",
            transform=ax.transAxes, va="top", ha="left", fontsize=7.5,
            family="monospace",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      alpha=0.85, edgecolor="lightgray"))

    ax.set_xticks(positions)
    ax.set_xticklabels(["30min", "60min"])
    ax.set_xlim(-0.6, 1.6)
    ax.set_ylabel(label)
    ax.set_title(label, fontsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ylim = YLIMS.get(col)
    if ylim is not None:
        ax.set_ylim(*ylim)
        all_pts = np.concatenate([x_v, y_v])
        all_pts = all_pts[np.isfinite(all_pts)]
        if all_pts.size > 0:
            lo_data = float(all_pts.min())
            hi_data = float(all_pts.max())
            if lo_data < ylim[0] or hi_data > ylim[1]:
                print(
                    f"[warn] {col}: data range [{lo_data:.3f}, {hi_data:.3f}] is clipped "
                    f"by y-limits [{ylim[0]}, {ylim[1]}]"
                )

fig.suptitle(f"Paired comparison: 30min vs 60min  (n={len(pair_keys)} flies)", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.97])

out_svg = DATA_DIR / "longvid_30min_60min_paired_comparison.svg"
out_png = DATA_DIR / "longvid_30min_60min_paired_comparison.png"
fig.savefig(out_svg, format="svg", bbox_inches="tight")
fig.savefig(out_png, dpi=150, bbox_inches="tight")
plt.close(fig)

print(f"\nsaved: {out_svg}")
print(f"saved: {out_png}")

# ---- Console table ----
print("\n==== Paired comparison: 30min vs 60min ====")
print(f"{'metric':<36s} {'Wilcoxon (W, p)':<28s} {'Paired-t (t, p)':<28s} {'n':>5s}")
print("-" * 100)
for label, w_stat, w_p, t_stat, t_p, n in stats_summary:
    if np.isfinite(w_p):
        wcol = f"W={w_stat:7.2f}, p={w_p:.4g}"
    else:
        wcol = "(insufficient)"
    if np.isfinite(t_p):
        tcol = f"t={t_stat:6.2f}, p={t_p:.4g}"
    else:
        tcol = "(insufficient)"
    print(f"{label:<36s} {wcol:<28s} {tcol:<28s} {n:>5d}")


Matched (date, fly) pairs with both 30min and 60min: 7
  2026_0423_Fly2
  2026_0427_Fly3
  2026_0429_Fly6
  2026_0429_Fly7
  2026_0505_Fly1
  2026_0505_Fly2
  2026_0505_Fly3

saved: C:\Users\Lylah\Desktop\data_processing\0_flight_30min_vs_60min\longvid_30min_60min_paired_comparison.svg
saved: C:\Users\Lylah\Desktop\data_processing\0_flight_30min_vs_60min\longvid_30min_60min_paired_comparison.png

==== Paired comparison: 30min vs 60min ====
metric                               Wilcoxon (W, p)              Paired-t (t, p)                  n
----------------------------------------------------------------------------------------------------
avg wingbeat amplitude (rad)         W=   1.00, p=0.03125         t=  2.89, p=0.02783              7
avg wingbeat frequency (Hz)          W=   1.00, p=0.03125         t=  3.16, p=0.01968              7
avg inferred flight power (W/kg)     W=   0.00, p=0.01562         t=  6.41, p=0.0006817            7
avg spiracle aperture                W=   0.00, p=0

### Block 5 — Pearson correlations: spiracle vs each flight signal

Within the same `[T_MIN_S, T_MAX_S]` window, `compute_correlations` computes the per-file Pearson r between spiracle aperture and each flight signal (returning `None` if the file has too few samples). Results are bucketed by condition and written to `longvid_30min_correlations.csv` / `longvid_60min_correlations.csv` (same shape as the summary CSVs), with a per-condition mean ± SEM printed to the console.

**中文**：在同一 `[T_MIN_S, T_MAX_S]` 窗内，`compute_correlations` 计算每个文件中气门开度与各飞行信号之间的皮尔逊相关系数 r（样本过少则返回 `None`）。结果按条件分桶并写入 `longvid_30min_correlations.csv` / `longvid_60min_correlations.csv`（结构与汇总 CSV 相同），并在控制台打印各条件的均值 ± SEM。

In [5]:
# Block 5 — Pearson correlations: spiracle size vs each other parameter
# Computed within the same [T_MIN_S, T_MAX_S] window used by Blocks 3/4.
# Saves two CSVs (30min / 60min) shaped like the summary CSVs:
#   first column = correlation label, subsequent columns = one per file.

CORR_TARGET = "rescaled_intensity"
CORR_PARTNERS = [
    ("wingbeat_amplitude",    "spiracle vs wingbeat amplitude"),
    ("wingbeat_frequency",    "spiracle vs wingbeat frequency"),
    ("inferred_flight_power", "spiracle vs inferred flight power"),
]
MIN_SAMPLES_FOR_CORR = 10


def compute_correlations(path: Path, t_min_s=None, t_max_s=None):
    """Pearson r of CORR_TARGET vs each partner within time_s in [t_min_s, t_max_s].

    Returns {partner_col: r} or None if the file is unusable.
    """
    df = pd.read_csv(path)
    if "time_s" not in df.columns or CORR_TARGET not in df.columns:
        return None
    if t_min_s is not None:
        df = df.loc[df["time_s"] >= t_min_s]
    if t_max_s is not None:
        df = df.loc[df["time_s"] <= t_max_s]
    if len(df) < MIN_SAMPLES_FOR_CORR:
        return None
    x = pd.to_numeric(df[CORR_TARGET], errors="coerce").values.astype(float)
    out = {}
    for col, _ in CORR_PARTNERS:
        if col not in df.columns:
            out[col] = np.nan
            continue
        y = pd.to_numeric(df[col], errors="coerce").values.astype(float)
        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() < MIN_SAMPLES_FOR_CORR:
            out[col] = np.nan
            continue
        xv = x[m]
        yv = y[m]
        if np.std(xv) == 0 or np.std(yv) == 0:
            out[col] = np.nan
            continue
        out[col] = float(np.corrcoef(xv, yv)[0, 1])
    return out


# (date, fly_num) -> list of (filename, {col: r})
corr_30 = {}
corr_60 = {}

# Re-glob so this block is self-contained (does not rely on Block 3 having run)
files_corr = sorted(
    DATA_DIR.rglob("*_combined.csv") if RECURSIVE else DATA_DIR.glob("*_combined.csv")
)
window_str = f"time_s in [{T_MIN_S}, {T_MAX_S}] s NIDAQ"
print(f"Pearson correlations on {len(files_corr)} file(s)  (window: {window_str})")

for p in files_corr:
    parsed = parse_basename(p.name)
    if parsed is None:
        continue
    date, fly_num, cond = parsed
    rs = compute_correlations(p, t_min_s=T_MIN_S, t_max_s=T_MAX_S)
    if rs is None:
        print(f"  [skip] {p.name}: insufficient data in window")
        continue
    bucket = corr_30 if cond == "30min" else corr_60
    bucket.setdefault((date, fly_num), []).append((p.name, rs))


def build_corr_table(bucket, condition_label: str) -> pd.DataFrame:
    """3-row DataFrame: 'correlation' column + one column per (date, fly)."""
    columns = {"correlation": [lbl for _, lbl in CORR_PARTNERS]}
    if not bucket:
        print(f"  [warn] No {condition_label} files for correlation")
        return pd.DataFrame(columns)
    for (date, fly_num) in sorted(bucket.keys()):
        entries = bucket[(date, fly_num)]
        col_name = f"{date}_Fly{fly_num}_{condition_label}"
        if len(entries) > 1:
            print(f"  [note] {condition_label}: {len(entries)} files match {date}_Fly{fly_num} -- averaging r values")
        col_vals = []
        for col, _ in CORR_PARTNERS:
            rs_vals = [rs[col] for _, rs in entries if col in rs and np.isfinite(rs[col])]
            col_vals.append(float(np.mean(rs_vals)) if rs_vals else np.nan)
        columns[col_name] = col_vals
    return pd.DataFrame(columns)


df_corr_30 = build_corr_table(corr_30, "30min")
df_corr_60 = build_corr_table(corr_60, "60min")

CORR_30_NAME = "longvid_30min_correlations.csv"
CORR_60_NAME = "longvid_60min_correlations.csv"
corr_30_path = DATA_DIR / CORR_30_NAME
corr_60_path = DATA_DIR / CORR_60_NAME
df_corr_30.to_csv(corr_30_path, index=False)
df_corr_60.to_csv(corr_60_path, index=False)

print(
    f"\nsaved: {corr_30_path}  ({df_corr_30.shape[0]} rows x {df_corr_30.shape[1]} cols, "
    f"{df_corr_30.shape[1] - 1} files)"
)
print(
    f"saved: {corr_60_path}  ({df_corr_60.shape[0]} rows x {df_corr_60.shape[1]} cols, "
    f"{df_corr_60.shape[1] - 1} files)"
)

# ---- Console summary: per-condition mean +/- SEM of each correlation ----
print("\n==== Mean Pearson r per condition ====")
print(f"{'correlation':<40s} {'30min (mean +/- SEM, n)':<30s} {'60min (mean +/- SEM, n)':<30s}")
print("-" * 110)
for col, label in CORR_PARTNERS:
    def _mean_sem(df):
        if df.empty:
            return np.nan, np.nan, 0
        row_idx = next(
            (i for i, lbl in enumerate(df["correlation"].values) if lbl == label),
            None
        )
        if row_idx is None:
            return np.nan, np.nan, 0
        row = df.iloc[row_idx, 1:].astype(float).values
        row = row[np.isfinite(row)]
        if len(row) == 0:
            return np.nan, np.nan, 0
        if len(row) == 1:
            return float(row[0]), 0.0, 1
        return float(row.mean()), float(row.std(ddof=1) / np.sqrt(len(row))), len(row)

    m30, s30, n30 = _mean_sem(df_corr_30)
    m60, s60, n60 = _mean_sem(df_corr_60)
    s30_str = f"{m30:+.3f} +/- {s30:.3f}, n={n30}" if n30 > 0 else "n=0"
    s60_str = f"{m60:+.3f} +/- {s60:.3f}, n={n60}" if n60 > 0 else "n=0"
    print(f"{label:<40s} {s30_str:<30s} {s60_str:<30s}")

Pearson correlations on 14 file(s)  (window: time_s in [10.0, 70.0] s NIDAQ)

saved: C:\Users\Lylah\Desktop\data_processing\0_flight_30min_vs_60min\longvid_30min_correlations.csv  (3 rows x 8 cols, 7 files)
saved: C:\Users\Lylah\Desktop\data_processing\0_flight_30min_vs_60min\longvid_60min_correlations.csv  (3 rows x 8 cols, 7 files)

==== Mean Pearson r per condition ====
correlation                              30min (mean +/- SEM, n)        60min (mean +/- SEM, n)       
--------------------------------------------------------------------------------------------------------------
spiracle vs wingbeat amplitude           +0.062 +/- 0.074, n=7          +0.120 +/- 0.140, n=7         
spiracle vs wingbeat frequency           +0.134 +/- 0.123, n=7          +0.193 +/- 0.114, n=7         
spiracle vs inferred flight power        +0.210 +/- 0.066, n=7          +0.204 +/- 0.130, n=7         


### Block 6 — 10–70 s overlay (30min vs 60min)

A 4×2 figure of raw time-series traces: rows = the four metrics, left column = all 30min traces overlaid, right column = all 60min traces. Each combined CSV is restricted to its own `[T_PLOT_MIN_S, T_PLOT_MAX_S]` window (independent of the analysis window used by Blocks 3–5). Saves `longvid_10_70s_overlay.svg/.png`.

**中文**：原始时间序列的 4×2 叠图：行 = 四个指标，左列 = 所有 30min 轨迹叠加，右列 = 所有 60min 轨迹。每个合并 CSV 都限制到各自的 `[T_PLOT_MIN_S, T_PLOT_MAX_S]` 绘图窗（与 Block 3–5 的分析窗相互独立）。保存为 `longvid_10_70s_overlay.svg/.png`。

In [6]:
# Block 6 - 10-70 s overlay: 30min vs 60min (4 metrics x 2 conditions)
# Reads every *_combined.csv in DATA_DIR, restricts to time_s in [T_PLOT_MIN_S, T_PLOT_MAX_S]
# (NIDAQ absolute time, inclusive), and produces a 4x2 figure: rows = metrics,
# left col = 30min overlay, right col = 60min overlay.
# Saves <OUT_NAME>.svg and .png in DATA_DIR. Independent of T_MIN_S / T_MAX_S used by Blocks 3-5.

import matplotlib.pyplot as plt

T_PLOT_MIN_S = 10.0
T_PLOT_MAX_S = 70.0
OUT_NAME     = f"longvid_{T_PLOT_MIN_S:.0f}_{T_PLOT_MAX_S:.0f}s_overlay"

# (column in <stem>_combined.csv, y-axis label, line color)
PLOT_METRICS = [
    ("rescaled_intensity",    "rescaled_intensity",            "steelblue"),
    ("wingbeat_amplitude",    "wingbeat_amplitude (rad)",      "mediumpurple"),
    ("wingbeat_frequency",    "wingbeat_frequency (Hz)",       "seagreen"),
    ("inferred_flight_power", "inferred_flight_power (W/kg)",  "darkorange"),
]

# Re-glob locally so the cell is self-contained.
files_plot = sorted(
    DATA_DIR.rglob("*_combined.csv") if RECURSIVE else DATA_DIR.glob("*_combined.csv")
)
print(
    f"Block 6: {len(files_plot)} combined CSV(s) under {DATA_DIR}  "
    f"(window: {T_PLOT_MIN_S:.1f} - {T_PLOT_MAX_S:.1f} s NIDAQ)"
)

# condition -> list of (label, df_window)
traces = {"30min": [], "60min": []}

for p in files_plot:
    parsed = parse_basename(p.name)
    if parsed is None:
        print(f"  [skip] {p.name} -- can't parse date/fly/condition")
        continue
    date, fly_num, cond = parsed
    try:
        df = pd.read_csv(p)
    except Exception as e:
        print(f"  [skip] {p.name}: {type(e).__name__}: {e}")
        continue
    if "time_s" not in df.columns:
        print(f"  [skip] {p.name}: no time_s column")
        continue
    df = df.loc[(df["time_s"] >= T_PLOT_MIN_S) & (df["time_s"] <= T_PLOT_MAX_S)].copy()
    if df.empty:
        print(f"  [skip] {p.name}: no rows within {T_PLOT_MIN_S:.1f} - {T_PLOT_MAX_S:.1f} s")
        continue
    df["time_s"] = pd.to_numeric(df["time_s"], errors="coerce")
    for col, _, _ in PLOT_METRICS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    traces[cond].append((f"{date}_Fly{fly_num}", df))

n30 = len(traces["30min"])
n60 = len(traces["60min"])
print(f"  30min traces: {n30}  |  60min traces: {n60}")

fig, axes = plt.subplots(4, 2, figsize=(12, 11), sharex="col", sharey="row")

CONDITIONS = [(0, "30min"), (1, "60min")]
for row_idx, (col_name, ylabel, color) in enumerate(PLOT_METRICS):
    for col_idx, cond_label in CONDITIONS:
        ax = axes[row_idx, col_idx]
        for trace_label, df in traces[cond_label]:
            if col_name not in df.columns:
                continue
            ax.plot(
                df["time_s"].values,
                df[col_name].values,
                color=color, linewidth=0.6, alpha=0.75, label=trace_label,
            )
        ax.grid(True, linestyle="--", alpha=0.35)
        if row_idx == 0:
            ax.set_title(cond_label, fontsize=12)
        if col_idx == 0:
            ax.set_ylabel(ylabel, fontsize=10)
        if row_idx == len(PLOT_METRICS) - 1:
            ax.set_xlabel("time_s (s, NIDAQ absolute)", fontsize=10)
            ax.set_xlim(T_PLOT_MIN_S, T_PLOT_MAX_S)
        if row_idx == 0 and traces[cond_label]:
            ax.legend(loc="upper right", fontsize=7, framealpha=0.85)

fig.suptitle(
    f"{T_PLOT_MIN_S:.0f} - {T_PLOT_MAX_S:.0f} s NIDAQ overlay: 30min vs 60min  "
    f"(n_30={n30}, n_60={n60})",
    fontsize=12,
)
fig.tight_layout(rect=[0, 0, 1, 0.96])

out_svg = DATA_DIR / f"{OUT_NAME}.svg"
out_png = DATA_DIR / f"{OUT_NAME}.png"
fig.savefig(out_svg, format="svg", bbox_inches="tight")
fig.savefig(out_png, dpi=150, bbox_inches="tight")
plt.close(fig)

print(f"\nsaved: {out_svg}")
print(f"saved: {out_png}")

Block 6: 14 combined CSV(s) under C:\Users\Lylah\Desktop\data_processing\0_flight_30min_vs_60min  (window: 10.0 - 70.0 s NIDAQ)
  30min traces: 7  |  60min traces: 7

saved: C:\Users\Lylah\Desktop\data_processing\0_flight_30min_vs_60min\longvid_10_70s_overlay.svg
saved: C:\Users\Lylah\Desktop\data_processing\0_flight_30min_vs_60min\longvid_10_70s_overlay.png
